# nl2sql demo

Notebook first, source files second. Each step shows one small change, then I can open the real function and explain why it exists.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

_here = Path.cwd().resolve()
ROOT = None
for candidate in (_here, _here.parent):
    if (candidate / 'nl2sql').exists():
        ROOT = candidate
        if str(candidate) not in sys.path:
            sys.path.append(str(candidate))
        break
if ROOT is None:
    raise RuntimeError('Could not find project root for notebook demo.')

from nl2sql.agent.agent_tools import schema_to_text
from nl2sql.agent.react_pipeline import _compact_execution_error, _parse_react_output
from nl2sql.core.llm import extract_first_select
from nl2sql.core.postprocess import guarded_postprocess
from nl2sql.core.prompting import make_few_shot_messages
from nl2sql.core.validation import validate_sql
from nl2sql.evaluation.eval import _build_item_pool, categorize_failure

print('demo helpers loaded')

This is the schema step.

If I open the file now, the thing I want to say is: this helper turns raw schema data into compact prompt text, because every condition needs the same grounding.

In [ ]:
# When I open agent_tools.py, I want to point at schema_to_text(...).
# Pair-programming phrasing: this function takes cached schema data in, and gives prompt text out.

schema_cache = {
    'tables': [
        {'name': 'customers', 'columns': [{'name': 'customerNumber'}, {'name': 'customerName'}, {'name': 'country'}]},
        {'name': 'orders', 'columns': [{'name': 'orderNumber'}, {'name': 'status'}, {'name': 'customerNumber'}]},
    ],
    'foreign_keys': [
        {'table': 'orders', 'column': 'customerNumber', 'ref_table': 'customers', 'ref_column': 'customerNumber'}
    ],
}

schema_text = schema_to_text(schema_cache)
print(schema_text)

This is the prompt step.

If I open the file now, the thing I want to say is: this file defines the prompt contract, and the main k=0 vs k=3 difference is just whether worked examples are inserted.

In [ ]:
# When I open prompting.py, I want to point at make_few_shot_messages(...).
# Pair-programming phrasing: the schema stays fixed, the real question stays fixed, and few-shot just adds worked examples in the middle.

exemplars = [
    {'nlq': 'List the customer names in France.', 'sql': "SELECT customerName FROM customers WHERE country = 'France';"},
    {'nlq': 'Count orders with status Shipped.', 'sql': "SELECT COUNT(*) FROM orders WHERE status = 'Shipped';"},
    {'nlq': 'Show each order number with its customer number.', 'sql': 'SELECT orderNumber, customerNumber FROM orders;'},
]

nlq = 'Show the customer names in France.'
messages_k0 = make_few_shot_messages(schema=schema_text, exemplars=[], nlq=nlq)
messages_k3 = make_few_shot_messages(schema=schema_text, exemplars=exemplars, nlq=nlq)

print('k=0 turns:', [m['role'] for m in messages_k0])
print('k=3 turns:', [m['role'] for m in messages_k3])
print('example question:', messages_k3[2]['content'])
print('example sql:', messages_k3[3]['content'])

This is the cleanup and guardrail step.

If I open the files now, the thing I want to say is: the system does conservative cleanup and simple validation before execution, rather than hiding mistakes with heavy repair.

In [ ]:
# When I open llm.py, postprocess.py, and validation.py, I want to show a straight input -> helper -> output path.
# Pair-programming phrasing: first recover the SQL, then clean it lightly, then make a simple valid/invalid decision.

raw = '''Here is the SQL:

SELECT customerName FROM customers WHERE country = 'France' ORDER BY customerName LIMIT 5;

This lists French customers.'''
extracted = extract_first_select(raw)
final_sql = guarded_postprocess(extracted or '', 'list customer names in France')

good_check = validate_sql(final_sql, schema_text)
bad_check = validate_sql('SELECT * FROM missing_table;', schema_text)

print('extracted:', extracted)
print('final:', final_sql)
print('valid case:', good_check)
print('invalid case:', bad_check)

This is the ReAct-only step.

If I open the file now, the thing I want to say is: the controller has to normalise messy model output into a stable action and SQL contract before it can loop.

In [ ]:
# When I open react_pipeline.py, I want to point at _parse_react_output(...).
# Pair-programming phrasing: this helper turns messy model text into something the controller can reliably act on.

strict = 'Thought: join the tables\nAction: query[SELECT customerName FROM customers;]'
fallback = 'I think the answer is:\nSELECT customerName FROM customers'
fail = 'I need more context before answering.'

for name, raw_case in [('strict', strict), ('fallback', fallback), ('fail', fail)]:
    _, action, sql = _parse_react_output(raw_case)
    print(f'{name}:', action, sql or '<none>')

print('feedback label:', _compact_execution_error("Unknown column 'o.total' in 'field list'"))

This is the evaluation step.

If I open the file now, the thing I want to say is: this is where fairness and correctness are defined, including leakage checks and failure labels.

In [ ]:
# When I open eval.py, I want to show that scoring is more than one accuracy number.
# Pair-programming phrasing: one helper protects the exemplar pool, and another keeps error analysis readable.

pred_rows = [('Alice',), ('Bob',)]
gold_rows = [('Bob',), ('Alice',)]

item = {'nlq': 'Show French customers', 'sql': "SELECT customerName FROM customers WHERE country = 'France';"}
pool = [
    item,
    {'nlq': 'Count shipped orders', 'sql': "SELECT COUNT(*) FROM orders WHERE status = 'Shipped';"},
]
filtered_pool = _build_item_pool(item=item, test_set=pool, exemplar_pool=pool, avoid_exemplar_leakage=True)
label = categorize_failure({'pred_sql': 'SELECT customerName FROM customers;', 'va': 1, 'ex': 0, 'error': ''})

print('EX idea:', Counter(pred_rows) == Counter(gold_rows))
print('pool size after leakage check:', len(filtered_pool))
print('failure label:', label)

These are the final submitted results.

If I open the CSVs or the analysis code now, the thing I want to say is: the dissertation tables come from the final pack, not from a live rerun in the notebook.

In [ ]:
# When I open simple_stats.py or build_final_analysis.py, I want to connect the code to the submitted tables.

summary = pd.read_csv(ROOT / 'results/final_analysis/summary_by_condition.csv')
pairwise = pd.read_csv(ROOT / 'results/final_analysis/pairwise_tests.csv')

print(summary[['condition_id', 'ex_mean']].to_string(index=False))
print()
print(pairwise[['comparison', 'p_value', 'decision_alpha_0_05']].to_string(index=False))

This is the code tour order.

If I get interrupted, I can come back to this list and keep the walkthrough moving.

In [ ]:
# This is my source-file roadmap.
# Pair-programming phrasing: I start with the shared path, then show the ReAct difference, then finish with scoring and stats.

roadmap = [
    ('schema_to_text / build_schema_summary', 'This is the shared schema grounding step.'),
    ('make_few_shot_messages', 'This is the prompt contract for k=0 and k=3.'),
    ('extract_first_select', 'This recovers SQL from noisy model output.'),
    ('guarded_postprocess', 'This keeps cleanup conservative.'),
    ('validate_sql', 'This is the simple pre-execution guardrail.'),
    ('_parse_react_output', 'This normalises raw ReAct output into action + SQL.'),
    ('run_react_pipeline', 'This is the bounded controller and last_good_sql logic.'),
    ('execution_accuracy / test_suite_accuracy_for_item', 'This is where correctness is defined.'),
    ('_build_item_pool / categorize_failure', 'This is leakage control and error analysis.'),
    ('compare_runs', 'This is the run-level Mann-Whitney test.'),
]

for name, note in roadmap:
    print(f'- {name}: {note}')